# Get the pathways of the selected genes
this is the inverse of the pathway analysis, I want to have a list of genes and add what are the pathways that are register for that gene

In [1]:
import scanpy as sc
import decoupler as dc

# Only needed for processing
import numpy as np
import pandas as pd
from upsetplot import generate_counts
from upsetplot import plot
from matplotlib import pyplot
from upsetplot import from_memberships
from upsetplot import from_contents


In [3]:
msigdb = pd.read_csv('../msigdb.csv')

In [20]:
sel_db = ['go_molecular_function',
                                  'go_cellular_component',
                                  'go_biological_process',
                                  'reactome_pathways',
                                  'kegg_pathways', 'hallmark']

In [11]:
msigdb.head()

,Unnamed: 0,genesymbol,collection,geneset
0,0,MAFF,chemical_and_genetic_perturbations,BOYAULT_LIVER_CANCER_SUBCLASS_G56_DN
1,1,MAFF,chemical_and_genetic_perturbations,ELVIDGE_HYPOXIA_UP
2,2,MAFF,chemical_and_genetic_perturbations,NUYTTEN_NIPP1_TARGETS_DN
3,3,MAFF,immunesigdb,GSE17721_POLYIC_VS_GARDIQUIMOD_4H_BMDC_DN
4,4,MAFF,chemical_and_genetic_perturbations,SCHAEFFER_PROSTATE_DEVELOPMENT_12HR_UP


In [19]:
msigdb['collection'].unique()

array(['chemical_and_genetic_perturbations', 'immunesigdb',
       'mirna_targets_mirdb', 'go_molecular_function', 'tf_targets_gtrf',
       'tf_targets_legacy', 'oncogenic_signatures',
       'cell_type_signatures', 'vaccine_response',
       'go_biological_process', 'cancer_gene_neighborhoods',
       'cancer_modules', 'go_cellular_component', 'wikipathways',
       'reactome_pathways', 'hallmark', 'mirna_targets_legacy',
       'biocarta_pathways', 'positional', 'human_phenotype_ontology',
       'pid_pathways', 'kegg_pathways'], dtype=object)

In [22]:
msigdb_filter =msigdb[msigdb['collection'].isin(sel_db)]

In [35]:
msigdb_immune =msigdb[msigdb['collection']=='immunesigdb']
msigdb_immune = msigdb_immune[~msigdb_immune['geneset'].str.contains('GSE')]
msigdb_immune[~msigdb_immune['geneset'].str.contains('_VS_')]

,Unnamed: 0,genesymbol,collection,geneset


In [6]:
gene_df = pd.read_csv("ortholog_selected_genes_sources.csv", index_col=0)

In [10]:
gene_df= gene_df[gene_df["caenorhabditis_elegans"].notna() & (gene_df["caenorhabditis_elegans"] != "")& (gene_df["caenorhabditis_elegans"] != " ")]
len(gene_df)

57

In [ ]:
geneset_dict = msigdb_filter.groupby("genesymbol")["geneset"].apply(list).to_dict()
geneset_dict

In [ ]:

# Add a new column 'Pathways' to merged_df by mapping genesymbol to genesets
gene_df["Pathways"] = gene_df["human"].map(geneset_dict)

gene_df

In [32]:
gene_df["n_Pathways"] = [len(p) if isinstance(p, list) else 0 for p in gene_df["Pathways"]]
gene_df.head()

,Ensembl,Shap/tstat,Source,LFC,Best,gorilla_gorilla,saccharomyces_cerevisiae,caenorhabditis_elegans,mus_musculus,pongo_abelii,human,Pathways,n_Pathways
Symbol,,,,,,,,,,,,,
NT5C2,ENSG00000076685.18,14.135628,Catboost 07,NaN,1.0,,,Y71H10B.1,Nt5c2,NT5C2,NT5C2,[GOBP_RIBONUCLEOSIDE_MONOPHOSPHATE_CATABOLIC_P...,57
ALDOA,ENSG00000285043.1,7.641741,Catboost 07,NaN,4.0,,,aldo-1,Aldoart1,,ALDOA,[GOBP_RIBONUCLEOSIDE_DIPHOSPHATE_METABOLIC_PRO...,103
BLCAP,ENSG00000166619.12,3.897527,Catboost 07,NaN,8.0,,,Y73E7A.6,Blcap,BLCAP,BLCAP,"[GOBP_APOPTOTIC_PROCESS, GOBP_CELLULAR_COMPONE...",6
FEZ2,ENSG00000171055.14,2.825766,Catboost 07,NaN,9.0,,,unc-76,Fez2,FEZ2,FEZ2,"[GOBP_TAXIS, GOBP_AUTOPHAGOSOME_ORGANIZATION, ...",35
SLC16A3,ENSG00000141526.16,1.558696,Catboost 07,NaN,13.0,,MCH5,mct-3,Slc16a3,SLC16A3,SLC16A3,"[GOCC_POSTSYNAPTIC_MEMBRANE, REACTOME_THE_CITR...",67


In [39]:
inmunne_gene_list=[]
with open("inmune_genes_symbol.txt", 'r') as file:
      inmunne_gene_list = [line.strip() for line in file] 


In [40]:
gene_df["immune_gene"] = ['immune' if g in inmunne_gene_list else '' for g in gene_df.index]


In [41]:
gene_df

,Ensembl,Shap/tstat,Source,LFC,Best,gorilla_gorilla,saccharomyces_cerevisiae,caenorhabditis_elegans,mus_musculus,pongo_abelii,human,Pathways,n_Pathways,immune_gene
Symbol,,,,,,,,,,,,,,
NT5C2,ENSG00000076685.18,14.135628,Catboost 07,NaN,1.0,,,Y71H10B.1,Nt5c2,NT5C2,NT5C2,[GOBP_RIBONUCLEOSIDE_MONOPHOSPHATE_CATABOLIC_P...,57,
ALDOA,ENSG00000285043.1,7.641741,Catboost 07,NaN,4.0,,,aldo-1,Aldoart1,,ALDOA,[GOBP_RIBONUCLEOSIDE_DIPHOSPHATE_METABOLIC_PRO...,103,immune
BLCAP,ENSG00000166619.12,3.897527,Catboost 07,NaN,8.0,,,Y73E7A.6,Blcap,BLCAP,BLCAP,"[GOBP_APOPTOTIC_PROCESS, GOBP_CELLULAR_COMPONE...",6,
FEZ2,ENSG00000171055.14,2.825766,Catboost 07,NaN,9.0,,,unc-76,Fez2,FEZ2,FEZ2,"[GOBP_TAXIS, GOBP_AUTOPHAGOSOME_ORGANIZATION, ...",35,
SLC16A3,ENSG00000141526.16,1.558696,Catboost 07,NaN,13.0,,MCH5,mct-3,Slc16a3,SLC16A3,SLC16A3,"[GOCC_POSTSYNAPTIC_MEMBRANE, REACTOME_THE_CITR...",67,
STUM,ENSG00000203685.9,1.490482,Catboost 07,NaN,14.0,,,Y51H7BR.7,Stum,STUM,STUM,NaN,0,
STUM,ENSG00000203685.9,19.534839,Catboost 40,NaN,1.0,,,Y51H7BR.7,Stum,STUM,STUM,NaN,0,
CA3,ENSG00000164879.6,4.181707,Catboost 40,NaN,5.0,,,cah-6,Car3,CA3,CA3,"[GOMF_PHOSPHATASE_ACTIVITY, GOMF_CARBON_OXYGEN...",17,
STIM1,ENSG00000167323.11,3.817382,Catboost 40,NaN,6.0,,,stim-1,Stim1,STIM1,STIM1,"[GOBP_CHEMICAL_HOMEOSTASIS, GOBP_REGULATION_OF...",109,immune


In [43]:
gene_df.to_csv("selected_genes_orthologs_c_elegans_pathways.csv")